# 02 - Hiperparámetros, Poda (Cost-Complexity Pruning) y Visualización

## Curso de Machine Learning: Modelos de Clasificación

Sin restricciones, un árbol de decisión continuará dividiéndose recursivamente hasta que cada hoja contenga una sola muestra (nodo completamente puro), alcanzando un Accuracy del $100\%$ en entrenamiento pero generalizando de forma pésima en datos reales.

En este cuaderno estudiaremos en profundidad cómo combatir este sobreajuste mediante **Pre-Poda (Early Stopping)** y **Post-Poda por Costo-Complejidad (Minimal Cost-Complexity Pruning - CCP)**, además de analizar la **Importancia de Características (Feature Importance)**.

---

### Contenidos
1. Sobreajuste en Árboles no Restringidos
2. Hiperparámetros de Pre-Poda (`max_depth`, `min_samples_split`, `min_samples_leaf`)
3. Post-Poda por Costo-Complejidad Matemática ($R_\alpha(T) = R(T) + \alpha |T|$)
4. Extracción de la Secuencia de Alfas (`cost_complexity_pruning_path`)
5. Selección de $\alpha$ Óptimo con Validación Cruzada
6. Importancia de Variables (Gini Importance MDI vs. Permutation Importance)
7. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Sobreajuste en Árboles no Restringidos

Entrenamos un árbol de decisión sin ninguna restricción de profundidad en el dataset `load_breast_cancer`:


In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Arbol completamente libre (overfitted)
tree_overfit = DecisionTreeClassifier(random_state=42)
tree_overfit.fit(X_train, y_train)

acc_train_overfit = tree_overfit.score(X_train, y_train)
acc_test_overfit = tree_overfit.score(X_test, y_test)

print(f"Profundidad alcanzada: {tree_overfit.get_depth()}")
print(f"Numero total de hojas: {tree_overfit.get_n_leaves()}")
print(f"Exactitud en Entrenamiento: {acc_train_overfit*100:.2f}% (Memorizacion completa)")
print(f"Exactitud en Prueba:        {acc_test_overfit*100:.2f}%")
print(f"Brecha de generalizacion:   {(acc_train_overfit - acc_test_overfit)*100:.2f}%")


## 2. Hiperparámetros de Pre-Poda

Scikit-Learn provee múltiples hiperparámetros para detener tempranamente el crecimiento:
- `max_depth`: Limita la profundidad máxima del árbol.
- `min_samples_split`: Número mínimo de muestras que debe contener un nodo para poder dividirse.
- `min_samples_leaf`: Número mínimo de muestras que debe conservar cualquier nodo hoja resultante.
- `max_leaf_nodes`: Límite superior sobre la cantidad total de hojas del árbol.


In [ ]:
# Efecto de max_depth sobre el error de entrenamiento vs validacion
profundidades = np.arange(1, 15)
train_scores = []
cv_scores = []

for d in profundidades:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_scores.append(clf.score(X_train, y_train))
    cv_scores.append(np.mean(cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy')))

plt.figure(figsize=(10, 5))
plt.plot(profundidades, train_scores, 'o-', color='#2b5c8f', label='Entrenamiento')
plt.plot(profundidades, cv_scores, 's-', color='#d95f02', label='Validacion Cruzada (5-Fold CV)')
plt.axvline(profundidades[np.argmax(cv_scores)], color='green', linestyle='--', label=f'Profundidad Optima: {profundidades[np.argmax(cv_scores)]}')
plt.xlabel('Profundidad Maxima (max_depth)', fontweight='bold')
plt.ylabel('Exactitud (Accuracy)', fontweight='bold')
plt.title('Diagnostico de Sobreajuste en Arboles: max_depth vs Desempeño', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 3. Post-Poda por Costo-Complejidad (Cost-Complexity Pruning)

La técnica matemática más rigurosa para podar un árbol es el **Minimal Cost-Complexity Pruning**. Consiste en penalizar la función de pérdida del árbol por su tamaño:

$$R_\alpha(T) = R(T) + \alpha |T|$$

Donde:
- $R(T)$: Es la tasa de error global del árbol $T$ en entrenamiento.
- $|T|$: Es el número de nodos hoja terminales de $T$.
- $\alpha \ge 0$: Es el hiperparámetro de penalización por complejidad (`ccp_alpha`).
  - Para $\alpha = 0$: Obtenemos el árbol completo original (sin poda).
  - A medida que $\alpha$ aumenta: Se eliminan progresivamente las ramas más débiles que aportan menor reducción de impureza, colapsándolas en nodos hoja.


In [ ]:
# 1. Obtener la trayectoria de podas (ccp_alphas) del arbol
path = tree_overfit.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

print(f"Total de valores candidatos de ccp_alpha generados: {len(ccp_alphas)}")
print(f"Primeros 5 alphas: {np.round(ccp_alphas[:5], 6)}")
print(f"Ultimos 3 alphas:  {np.round(ccp_alphas[-3:], 6)}")


In [ ]:
# 2. Entrenar arboles para cada valor de alpha candidato
clfs = []
for alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    clf.fit(X_train, y_train)
    clfs.append(clf)

# Remover el ultimo clasificador (que tiene 1 solo nodo hoja)
clfs = clfs[:-1]
ccp_alphas = ccp_alphas[:-1]

# 3. Graficar Exactitud vs Alpha para entrenamiento y prueba
train_acc = [c.score(X_train, y_train) for c in clfs]
test_acc = [c.score(X_test, y_test) for c in clfs]

plt.figure(figsize=(10, 5))
plt.plot(ccp_alphas, train_acc, marker='o', label='Exactitud Entrenamiento', color='#2b5c8f', drawstyle='steps-post')
plt.plot(ccp_alphas, test_acc, marker='s', label='Exactitud Prueba', color='#d95f02', drawstyle='steps-post')

idx_best = np.argmax(test_acc)
best_alpha = ccp_alphas[idx_best]

plt.axvline(best_alpha, color='green', linestyle='--', label=f'Alpha Optimo: {best_alpha:.5f}')
plt.xlabel('Parametro de Penalizacion ccp_alpha', fontweight='bold')
plt.ylabel('Exactitud', fontweight='bold')
plt.title('Seleccion de Poda Optima por Costo-Complejidad (CCP)', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mejor Exactitud en Prueba con Poda CCP: {test_acc[idx_best]*100:.2f}% (vs {acc_test_overfit*100:.2f}% del arbol sin podar)")


## 4. Importancia de Variables: MDI vs. Permutación

Scikit-Learn calcula automáticamente `feature_importances_` basado en **MDI (Mean Decrease in Impurity)**: la suma ponderada de las reducciones de impureza Gini aportadas por cada variable en todas las divisiones.

Sin embargo, MDI presenta un sesgo conocido: **sobreestima la importancia de variables con alta cardinalidad o valores continuos con mucho ruido**. Para validar la importancia real, se recomienda contrastar con **Permutation Importance**.


In [ ]:
# Arbol podado optimo
tree_podado = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha).fit(X_train, y_train)

# 1. Importancia MDI
importancias_mdi = pd.Series(tree_podado.feature_importances_, index=cancer.feature_names)
top_mdi = importancias_mdi.sort_values(ascending=False).head(8)

# 2. Permutation Importance
perm_importance = permutation_importance(tree_podado, X_test, y_test, n_repeats=10, random_state=42)
top_perm = pd.Series(perm_importance.importances_mean, index=cancer.feature_names).sort_values(ascending=False).head(8)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

top_mdi.plot(kind='barh', ax=axes[0], color='#2b5c8f')
axes[0].set_title('Top 8 Variables - MDI (Gini Importance)', fontweight='bold')
axes[0].set_xlabel('Reduccion Total de Impureza')
axes[0].invert_yaxis()

top_perm.plot(kind='barh', ax=axes[1], color='#2ca02c')
axes[1].set_title('Top 8 Variables - Permutation Importance (Test)', fontweight='bold')
axes[1].set_xlabel('Caida Media en Exactitud')
axes[1].invert_yaxis()

plt.suptitle('Comparativa de Metodos de Interpretabilidad en Arboles Podados', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Conclusiones

1. **Sobreajuste Natural**: Los árboles crecen hasta memorizar el conjunto de entrenamiento si no se controlan.
2. **Pre-Poda**: Rápida e intuitiva (`max_depth=3` a `5` suele ser un buen punto de partida).
3. **Poda CCP (`ccp_alpha`)**: El enfoque matemáticamente más elegante; poda el árbol podando las ramas con menor ganancia neta.
4. **Interpretabilidad**: La importancia de variables debe contrastarse con permutaciones para evitar sesgos numéricos.
